In [1]:
# 1. Setup and corpus
import numpy as np
import pandas as pd

corpus = [
    "he started driving when he was 22 years old",
    "she was fed up of the assignments",
    "he screwed up the exam",
    "he did not like trekking",
    "he could not bear the pain of separation"
]

tokenized = [s.lower().split() for s in corpus]
vocab = sorted(set(w for s in tokenized for w in s))
vocab_index = {w: i for i, w in enumerate(vocab)}
print(vocab)


['22', 'assignments', 'bear', 'could', 'did', 'driving', 'exam', 'fed', 'he', 'like', 'not', 'of', 'old', 'pain', 'screwed', 'separation', 'she', 'started', 'the', 'trekking', 'up', 'was', 'when', 'years']


In [2]:
# 2. One-Hot Encoding (OHE)
def one_hot_encode(sentence_tokens, vocab_index):
    matrix = np.zeros((len(sentence_tokens), len(vocab_index)), dtype=int)
    for row, word in enumerate(sentence_tokens):
        matrix[row, vocab_index[word]] = 1
    return matrix

ohe_matrix = one_hot_encode(tokenized[0], vocab_index)
ohe_df = pd.DataFrame(ohe_matrix, index=tokenized[0], columns=vocab)
ohe_df


,22,assignments,bear,could,did,driving,exam,fed,he,like,...,screwed,separation,she,started,the,trekking,up,was,when,years
he,0,0,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
started,0,0,0,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0
driving,0,0,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
when,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
he,0,0,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
was,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
22,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
years,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
old,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [3]:
# 3. Bag of Words (BoW)
def bag_of_words(tokenized_corpus, vocab_index, binary=False):
    matrix = np.zeros((len(tokenized_corpus), len(vocab_index)), dtype=int)
    for row, sentence in enumerate(tokenized_corpus):
        for word in sentence:
            col = vocab_index[word]
            if binary:
                matrix[row, col] = 1
            else:
                matrix[row, col] += 1
    return matrix

bow_matrix = bag_of_words(tokenized, vocab_index)
bow_df = pd.DataFrame(bow_matrix, columns=vocab)
bow_df


,22,assignments,bear,could,did,driving,exam,fed,he,like,...,screwed,separation,she,started,the,trekking,up,was,when,years
0,1,0,0,0,0,1,0,0,2,0,...,0,0,0,1,0,0,0,1,1,1
1,0,1,0,0,0,0,0,1,0,0,...,0,0,1,0,1,0,1,1,0,0
2,0,0,0,0,0,0,1,0,1,0,...,1,0,0,0,1,0,1,0,0,0
3,0,0,0,0,1,0,0,0,1,1,...,0,0,0,0,0,1,0,0,0,0
4,0,0,1,1,0,0,0,0,1,0,...,0,1,0,0,1,0,0,0,0,0


In [4]:
# 4. TF-IDF Embedding
def compute_tf(tokenized_corpus, vocab_index):
    tf = np.zeros((len(tokenized_corpus), len(vocab_index)))
    for row, sentence in enumerate(tokenized_corpus):
        for word in sentence:
            tf[row, vocab_index[word]] += 1
        tf[row] /= len(sentence)
    return tf

def compute_idf(tokenized_corpus, vocab_index):
    n_docs = len(tokenized_corpus)
    df = np.zeros(len(vocab_index))
    for sentence in tokenized_corpus:
        for word in set(sentence):
            df[vocab_index[word]] += 1
    return np.log(n_docs / df)

tf_matrix = compute_tf(tokenized, vocab_index)
idf_vector = compute_idf(tokenized, vocab_index)
tfidf_matrix = tf_matrix * idf_vector
tfidf_df = pd.DataFrame(tfidf_matrix, columns=vocab)
tfidf_df.round(3)


,22,assignments,bear,could,did,driving,exam,fed,he,like,...,screwed,separation,she,started,the,trekking,up,was,when,years
0,0.179,0.00,0.000,0.000,0.000,0.179,0.000,0.00,0.050,0.000,...,0.000,0.000,0.00,0.179,0.000,0.000,0.000,0.102,0.179,0.179
1,0.000,0.23,0.000,0.000,0.000,0.000,0.000,0.23,0.000,0.000,...,0.000,0.000,0.23,0.000,0.073,0.000,0.131,0.131,0.000,0.000
2,0.000,0.00,0.000,0.000,0.000,0.000,0.322,0.00,0.045,0.000,...,0.322,0.000,0.00,0.000,0.102,0.000,0.183,0.000,0.000,0.000
3,0.000,0.00,0.000,0.000,0.322,0.000,0.000,0.00,0.045,0.322,...,0.000,0.000,0.00,0.000,0.000,0.322,0.000,0.000,0.000,0.000
4,0.000,0.00,0.201,0.201,0.000,0.000,0.000,0.00,0.028,0.000,...,0.000,0.201,0.00,0.000,0.064,0.000,0.000,0.000,0.000,0.000


In [5]:
# 5. Cosine similarity helper
def cosine_similarity(v1, v2):
    return np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))

sim = cosine_similarity(tfidf_matrix[2], tfidf_matrix[3])
sim


np.float64(0.0067246623421990635)

In [6]:
# 6. Word2Vec (neural embedding)
from gensim.models import Word2Vec

w2v_model = Word2Vec(
    sentences=tokenized,
    vector_size=50,
    window=3,
    min_count=1,
    sg=1,
    epochs=200
)

word_vector = w2v_model.wv["he"]
word_vector.shape


(50,)

In [7]:
# 7. Word2Vec similar words
w2v_model.wv.most_similar("he", topn=5)


[('old', 0.34844768047332764),
 ('driving', 0.30932721495628357),
 ('screwed', 0.303088515996933),
 ('like', 0.29817405343055725),
 ('started', 0.2882643938064575)]